In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib
import warnings
warnings.filterwarnings('ignore')

In [2]:
X_train = pd.read_csv('../Datos/preparados/TrainX.csv')
y_train = pd.read_csv('../Datos/preparados/TrainY.csv')
X_val   = pd.read_csv('../Datos/preparados/ValidationX.csv')
y_val   = pd.read_csv('../Datos/preparados/ValidationY.csv')
X_test  = pd.read_csv('../Datos/preparados/TestX.csv')
y_test  = pd.read_csv('../Datos/preparados/TestY.csv')

y_train = y_train.values.ravel()
y_val   = y_val.values.ravel()
y_test  = y_test.values.ravel()


print("Shapes -> X_train:", X_train.shape, "X_val:", X_val.shape, "X_test:", X_test.shape)

Shapes -> X_train: (461100, 2) X_val: (115276, 2) X_test: (144095, 2)


PCA es un modelo de clasificación y no de regresion, asi que para poder usar este modelo debo pasar roi a categorias que es lo que hago enl a siguiente parte

In [3]:
p33, p66 = np.percentile(y_train, [33, 66])
def to_classes(y): 
    return np.digitize(y, bins=[p33, p66])

y_train_cl = to_classes(y_train)
y_val_cl   = to_classes(y_val)
y_test_cl  = to_classes(y_test)

print(f"Cortes 33%/66% (desde train): {p33:.6f}, {p66:.6f}")
print("Distribución clases (train/val/test):",
      np.bincount(y_train_cl), np.bincount(y_val_cl), np.bincount(y_test_cl))

Cortes 33%/66% (desde train): 0.579860, 2.474789
Distribución clases (train/val/test): [152163 152163 156774] [38086 38063 39127] [47820 47379 48896]


In [5]:
solvers = ['svd', 'lsqr', 'eigen']
shrinkages = [None, 'auto']
tols = [0.0001, 0.001, 0.01]
resultados = []

OUT_DIR = '../experimentos/resultados_PCA'
os.makedirs(OUT_DIR, exist_ok=True)

best_error = 1.0 + 1e-9
best_params = None


for solver in solvers:
    for shrink in shrinkages:
        if solver == 'svd' and shrink == 'auto':
            continue
        for tol in tols:
            try:
                model = LinearDiscriminantAnalysis(solver=solver, shrinkage=shrink, tol=tol)
                model.fit(X_train, y_train_cl)

                y_tr_pred = model.predict(X_train)
                y_val_pred = model.predict(X_val)

                err_tr = 1.0 - accuracy_score(y_train_cl, y_tr_pred)
                err_val = 1.0 - accuracy_score(y_val_cl, y_val_pred)

                resultados.append({
                    'solver': solver,
                    'shrinkage': 'None' if shrink is None else str(shrink),
                    'tol': tol,
                    'error_train': err_tr,
                    'error_val': err_val
                })

                
                if err_val < best_error:
                    best_error = err_val
                    best_params = {'solver': solver, 'shrinkage': shrink, 'tol': tol}
            except Exception as e: #si falla en algu na combinacion
                print(f"Fallo combinación solver={solver}, shrink={shrink}, tol={tol} -> {e}")


In [6]:
df_res = pd.DataFrame(resultados).sort_values('error_val').reset_index(drop=True)
df_res.to_csv(os.path.join(OUT_DIR, 'tabla_resultados_PCA.csv'), index=False)
print("\nTabla de resultados guardada en:", os.path.join(OUT_DIR, 'tabla_resultados_PCA.csv'))

print("\nTop 3 combinaciones (menor error_val):")
print(df_res.head(3))


Tabla de resultados guardada en: ../experimentos/resultados_LDA\tabla_resultados_LDA.csv

Top 3 combinaciones (menor error_val):
  solver shrinkage     tol  error_train  error_val
0    svd      None  0.0001     0.509729   0.511598
1    svd      None  0.0010     0.509718   0.511598
2    svd      None  0.0100     0.509718   0.511598


In [7]:
#entrenar con los mejores hiperparametros
if best_params is None:
    raise RuntimeError("No se encontró ningún modelo válido durante la búsqueda. Revisa los datos o los parámetros.")

print("\nMejores hiperparámetros seleccionados:", best_params, "con error_val =", best_error)
pca_best = LinearDiscriminantAnalysis(
    solver=best_params['solver'],
    shrinkage=best_params['shrinkage'],
    tol=best_params['tol']
)
pca_best.fit(X_train, y_train_cl)


Mejores hiperparámetros seleccionados: {'solver': 'svd', 'shrinkage': None, 'tol': 0.0001} con error_val = 0.5115982511537527


LinearDiscriminantAnalysis()

In [8]:
#predecir el test
y_test_pred = pca_best.predict(X_test)
error_test = 1.0 - accuracy_score(y_test_cl, y_test_pred)

print(f"\nError en TEST: {error_test:.4f}")
print("\nReporte clasificación (TEST):")
print(classification_report(y_test_cl, y_test_pred))
print("Matriz de confusión (TEST):")
print(confusion_matrix(y_test_cl, y_test_pred))


Error en TEST: 0.5096

Reporte clasificación (TEST):
              precision    recall  f1-score   support

           0       0.52      0.31      0.39     47820
           1       0.42      0.28      0.34     47379
           2       0.51      0.86      0.64     48896

    accuracy                           0.49    144095
   macro avg       0.48      0.49      0.46    144095
weighted avg       0.48      0.49      0.46    144095

Matriz de confusión (TEST):
[[14991 15691 17138]
 [ 9973 13398 24008]
 [ 3707  2917 42272]]
